Diagonalizing the Hamiltonian matrix for the LTFIM hamiltonian to find the energy eigenvalues and eigenkets. Calculate the groundstate magnetization and energy at zero temperature and finite temperature.

$$
H = -J\sum_{\langle i j \rangle} \sigma^z_i \sigma^z_j- h \sum_{i=1}^N \sigma^x_i - \Omega \sum_{i=1}^N \sigma^z_i
$$
where ${\bf \sigma}_i$ are Pauli operators.

In [1]:
using LinearAlgebra

N = 4
Dim = 2^N

B = 1.
Ω = 1.
J = 1.

Hamiltonian = zeros(Dim,Dim)   #This is your 2D Hamiltonian matrix  

for Ket = 0:Dim-1  #Loop over Hilbert Space
    Diagonal = 0.
    # bond term
    for SpinIndex = 0:N-2  #Loop over spin index (base zero, stop one spin before the end of the chain)
        Spin1 = 2*((Ket>>SpinIndex)&1) - 1
        NextIndex = SpinIndex + 1
        Spin2 = 2*((Ket>>NextIndex)&1) - 1
        Diagonal = Diagonal - J*Spin1*Spin2 #spins are +1 and -1
    end
    
    # long. field term
    for SpinIndex = 0:N-1
        Spin = 2*((Ket>>SpinIndex)&1) - 1
        Diagonal = Diagonal - Ω*Spin
    end
    
    Hamiltonian[Ket+1,Ket+1] = Diagonal
    
    for SpinIndex = 0:N-1
        bit = 2^SpinIndex   #The "label" of the bit to be flipped
        Bra = Ket ⊻ bit    #Binary XOR flips the bit
        Hamiltonian[Bra+1,Ket+1] = -B
    end
    
end

In [2]:
### ZERO-T GROUND STATE ###

Diag = eigen(Hamiltonian)     #Diagonalize the Hamiltonian
GroundState = Diag.vectors[:, 1];  #this gives the groundstate eigenvector
E0 = Diag.values[1] / N
println("E0 / N = ",E0)

magnetization = 0
for Ket = 0:Dim-1  #Loop over Hilbert Space
    SumSz = 0.
    for SpinIndex = 0:N-1  #Loop over spin index (base zero, stop one spin before the end of the chain)
        Spin1 = 2*((Ket>>SpinIndex)&1) - 1
        SumSz += Spin1 #spin is +1 or -1
    end
    magnetization += SumSz*SumSz*GroundState[Ket+1]^2  #Don't forgot to square the coefficients...
end

println("Mag. / N^2 = ", magnetization/(N*N))

E0 / N = -1.9549727175737188
Mag. / N^2 = 0.8758680175906756


In [3]:
### FINITE-T ###

β = 1
    
expH = exp(-β*Hamiltonian)
Z = tr(expH) # partition function
ρ = expH / Z

sigmaZ1 = [1 0; 0 -1]
sigmaZN = kron(sigmaZ1, sigmaZ1)
for i = 3:N
    sigmaZN = kron(sigmaZN, sigmaZ1)
end

m_thermal = tr(sigmaZN*ρ) / (N)
E_thermal = tr(Hamiltonian*ρ) / N

println("E thermal / N = ", E_thermal)
println("Mag. / N = ", m_thermal)

E thermal / N = -1.907862225728211
Mag. / N = 0.1660975117531079
